# Quick Submission

In this notebook, we use the cleaned up data to make submission to the original [House Prices competition](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques). The majority of the preprocessing is already done, so we can do our modeling and feature engineering with minimal code.

In [1]:
# Global Variables
NUM_FOLDS = 12
RANDOM_SEED = 10

# Imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Original Data
original_train = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv', index_col = 'Id')
original_test = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv', index_col = 'Id')
submission = pd.read_csv('../input/house-prices-advanced-regression-techniques/sample_submission.csv')

# Preprocessed Data
train = pd.read_csv('../input/house-prices-ames-cleaned-dataset/new_train.csv', index_col = 'Id')
test = pd.read_csv('../input/house-prices-ames-cleaned-dataset/new_test.csv', index_col = 'Id')

# Feature Engineering

## 1. Indicator Variables

Binary-valued columns which indicate if a house has a certain feature or not.

In [2]:
# Has MiscFeature
train['MiscFeature'] = (train['MiscFeature'].notna()).astype(int)
test['MiscFeature'] = (test['MiscFeature'].notna()).astype(int)

# Has Garage
train['HasGarage'] = (train['GarageType'].notna()).astype(int)
test['HasGarage'] = (test['GarageType'].notna()).astype(int)

# Has Pool
train['HasPool'] = (train['PoolArea'] > 0).astype(int)
test['HasPool'] = (test['PoolArea'] > 0).astype(int)
train.drop(columns = 'PoolQC', inplace = True)
test.drop(columns = 'PoolQC', inplace = True)

# Has Porch/Deck
porch_cols = ['WoodDeckSF','OpenPorchSF','EnclosedPorch','3SsnPorch','ScreenPorch']
train['HasPorch'] = (train[porch_cols].sum(axis = 1) > 0).astype(int)
test['HasPorch'] = (test[porch_cols].sum(axis = 1) > 0).astype(int)

# Has Fireplace
train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)
test['HasFireplace'] = (test['Fireplaces'] > 0).astype(int)

# Has Fence
train['HasFence'] = (train['Fence'] > 0).astype(int)
test['HasFence'] = (test['Fence'] > 0).astype(int)

# Has Veneer
train['HasVeneer'] = (train['MasVnrType'].notna()).astype(int)
test['HasVeneer'] = (test['MasVnrType'].notna()).astype(int)

# Has 2nd Floor
train['Has2ndFloor'] = (train['2ndFlrSF'] > 0).astype(int)
test['Has2ndFloor'] = (test['2ndFlrSF'] > 0).astype(int)

# Has Basement
train['HasBasement'] = (train['BsmtCond'] > 0).astype(int)
test['HasBasement'] = (test['BsmtCond'] > 0).astype(int)

# Was Remodeled
train['Remodel'] = (train['YearRemodAdd'] != train['YearBuilt']).astype(int)
test['Remodel'] = (test['YearRemodAdd'] != test['YearBuilt']).astype(int)

## 2. Transformations

Combining multiple columns arithmetically to form new features 

In [3]:
# Age
train['House_Age'] = train['YrSold'] - train['YearBuilt']
test['House_Age'] = test['YrSold'] - test['YearBuilt']

# Total Square Footage
sf_cols = ['TotalBsmtSF','1stFlrSF','2ndFlrSF']
train['TotalSF'] = train[sf_cols].sum(axis = 1)
test['TotalSF'] = test[sf_cols].sum(axis = 1)

# Total bathrooms
train['TotalBath'] = train[['FullBath','BsmtFullBath']].sum(axis = 1) + 0.5 * train[['HalfBath','BsmtHalfBath']].sum(axis = 1)
test['TotalBath'] = test[['FullBath','BsmtFullBath']].sum(axis = 1) + 0.5 * test[['HalfBath','BsmtHalfBath']].sum(axis = 1)

# Total Porch/Deck space
train['TotalPorch'] = train[porch_cols].sum(axis = 1)
test['TotalPorch'] = test[porch_cols].sum(axis = 1)

# Ratio of living space to Lot Size
train["LivLotRatio"] = train["GrLivArea"] / train["LotArea"]
test["LivLotRatio"] = test["GrLivArea"] / test["LotArea"]

# Avg square feet per room
train["Spaciousness"] = (train["1stFlrSF"]+train["2ndFlrSF"]) / train["TotRmsAbvGrd"]
test["Spaciousness"] = (test["1stFlrSF"]+test["2ndFlrSF"]) / test["TotRmsAbvGrd"]

# Lot space + frontage
train['TotalLot'] = train['LotFrontage'] + train['LotArea']
test['TotalLot'] = test['LotFrontage'] + test['LotArea']

# Total finished basement
train['TotalBsmtFin'] = train['BsmtFinSF1'] + train['BsmtFinSF2']
test['TotalBsmtFin'] = test['BsmtFinSF1'] + test['BsmtFinSF2']

# PCA inspired feature 
train["PCA_Feature1"] = train['GrLivArea'] + train['TotalBsmtSF']
test["PCA_Feature1"] = test['GrLivArea'] + test['TotalBsmtSF']

# PCA inspired feature
train["PCA_Feature2"] = train['YearRemodAdd'] * train['TotalBsmtSF']
test["PCA_Feature2"] = test['YearRemodAdd'] * test['TotalBsmtSF']

temp = train.isna().sum() 
display(temp[temp > 0])
temp = test.isna().sum() 
display(temp[temp > 0])

LotFrontage     259
Alley          1365
MasVnrType        8
GarageType       81
TotalLot        259
dtype: int64

LotFrontage     227
Alley          1352
MasVnrType       15
GarageType       77
TotalLot        227
dtype: int64

## 3. Preprocessing

Before we can start training models, we still need to encode the remaining categorical data and fill in the missing values.

* Categorical variables - one hot encode (NAs becomes 0)
* Ordered variables - MinMaxScaler -> `0 < x < 1`
* Skewed variables - Log transform (NAs replaced with the mean) then normalize
* Numerical variables - Normalize (NAs replaced with the mean)

In [4]:
# Preprocessing Imports
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import RobustScaler, MinMaxScaler, FunctionTransformer, OneHotEncoder
from category_encoders import OneHotEncoder

In [5]:
# Categorical Preprocessing
features = list(test.columns)
categorical = [x for x in features if test[x].dtype == 'object']
cat_pipeline = OneHotEncoder()

# Ordinal Preprocessing
ordinal = [x for x in features if x not in categorical and len(test[x].unique()) <= 10]
numerical = [x for x in features if x not in ordinal and x not in categorical]
ord_pipeline = make_pipeline(SimpleImputer(), RobustScaler())

# Skewed Variable Preprocessing (skewed/not skewed)
temp = train[numerical].skew()
skewed = list(temp[temp > 0.7].index)
skew_pipeline = make_pipeline(SimpleImputer(), FunctionTransformer(np.log1p), RobustScaler())

# Numerical Variable Preprocessing (not skewed)
num_pipeline = make_pipeline(SimpleImputer(), RobustScaler())

# Full Preprocessing Pipeline
pipeline = make_column_transformer(
    (cat_pipeline, categorical),
    (ord_pipeline, ordinal),
    (skew_pipeline, skewed),
    remainder = num_pipeline
)

# Submission

We create a simple submission to the original competition. The main features of our submission are as follows:

1. Taking the log of `SalePrice` for fitting the model
2. Each CV fold has the same ratio of houses in each `Neighborhood`
3. Average test predictions over each fold

In [6]:
# Modeling Imports
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_log_error, mean_squared_error

In [7]:
%%time

# Cross Validation
scores = cross_validate(
    estimator = make_pipeline(pipeline, Lasso(alpha = 0.001)),
    X = train[test.columns],
    y = np.log1p(train['SalePrice']),
    scoring = 'neg_root_mean_squared_error',
    cv = StratifiedKFold(n_splits = NUM_FOLDS, shuffle = True, random_state = RANDOM_SEED).split(train, train['Neighborhood']),
    return_estimator = True
)

# Submission
preds = np.zeros(len(test))
for fold, (score, estimator) in enumerate(zip(scores['test_score'], scores['estimator']), start = 1):
    preds += estimator.predict(test) / NUM_FOLDS # Avg over each fold
    print(f'Fold {fold} (RSME):', round(-score, 6))
submission['SalePrice'] = np.expm1(preds)
submission.to_csv('submission.csv', index = False,)

print('\nBest    (RSME):', round(-np.max(scores['test_score']), 6))
print('Median  (RSME):', round(-np.mean(scores['test_score']), 6))
print('Average (RSME):', round(-np.median(scores['test_score']), 6))
print('Worst   (RSME):', round(-np.min(scores['test_score']), 6))

Fold 1 (RSME): 0.110825
Fold 2 (RSME): 0.104334
Fold 3 (RSME): 0.1015
Fold 4 (RSME): 0.134326
Fold 5 (RSME): 0.096613
Fold 6 (RSME): 0.123613
Fold 7 (RSME): 0.094804
Fold 8 (RSME): 0.115165
Fold 9 (RSME): 0.11203
Fold 10 (RSME): 0.100703
Fold 11 (RSME): 0.132779
Fold 12 (RSME): 0.098158

Best    (RSME): 0.094804
Median  (RSME): 0.110404
Average (RSME): 0.107579
Worst   (RSME): 0.134326
CPU times: user 14.7 s, sys: 10.2 s, total: 24.9 s
Wall time: 8.94 s
